In [ ]:
!pip install git+https://github.com/huggingface/transformers.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip install torch tensorboard
!pip install datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub scipy einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import os
import json
import urllib.request
from typing import Dict, List, Optional
import torch
from datasets import Dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
token = os.environ.get("HF_TOKEN")
login(token=token)

In [ ]:
MODEL_NAME = "google/gemma-4-E2B-it"
DATA_URL = "https://raw.githubusercontent.com/tangqiaoyu/ToolAlpaca/main/data/train_data.json"

In [ ]:
LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.05
MAX_SEQ_LEN = 256
TRAIN_BATCH = 2
GRAD_ACCUM = 8
LR = 1e-4
NUM_EPOCHS = 2
MAX_TRAIN_STEPS = 500

In [ ]:
def download_json(url, local_path):
    if not os.path.exists(local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        urllib.request.urlretrieve(url, local_path)

    with open(local_path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
DATA_PATH = "/content/train_data.json"
raw_train = download_json(DATA_URL, DATA_PATH)

In [ ]:
raw_train[0]

{'Name': 'Httpbin',
 'Description': 'A Simple HTTP Request & Response Service',
 'Link': 'https://httpbin.org/',
 'Category': 'Development',
 'Introduction': "The Httpbin API (Application Programming Interface) is a simple HTTP request and response service that enables developers to test their applications' interactions with the HTTP protocol. Httpbin provides the following functions: 1) Basic GET, POST, PUT, DELETE, HEAD, and PATCH http request testing. 2) Retrieval of client request data, including headers, form data, uploaded files, and cookies. 3) Endpoint for proxy headers testing. 4) Ability to simulate status codes and other responses. Httpbin is an easy-to-use, efficient, and reliable tool for testing and debugging HTTP-related issues in web and mobile applications.",
 'Functions': '1. Name: sendHttpRequest\nDescription: Send an HTTP request with the specified method, headers, and data to the Httpbin API for testing purposes.\nInput: {"method": "Required. String. The HTTP metho

In [ ]:
SYSTEM_PROMPT = """You are an intelligent AI agent that answers questions by reasoning step-by-step.

You have access to the following tools:
{tool_docs}

Use this STRICT format for reasoning — output a JSON block:
{{"thought": "...", "action": "...", "action_input": {{"key": "value"}}}}

If you have enough information, output a final JSON block:
{{"thought": "...", "final_answer": "..."}}

RULES:
- STOP generating immediately after closing the JSON block.
- action_input MUST be a valid JSON object (double-quoted keys, no trailing commas).
- Only call tools listed above; never fabricate tool names or results.
- Always output valid JSON — no trailing commas, no single quotes.
- Use the final_answer JSON format (with "thought" and "final_answer" keys, omitting "action" and "action_input") only when you have gathered enough information to fully answer the user's question."""

In [ ]:
def parse_output_trajectory_reasoning(intermediate_step):

    step = {}
    tool_name, action_input, thought_action = intermediate_step[0]

    step['thought'] = thought_action
    step['action'] = tool_name
    step['action_input'] = action_input

    return json.dumps(step)

In [ ]:
def parse_output_trajectory_answer(instance):

    step = {}

    final_thought = instance.get("Final Thought", "")
    output = instance.get("output", "")

    step['thought'] = final_thought
    step['final_answer'] = output

    return json.dumps(step)

In [ ]:
def make_training_example(tool, instance, step):

    history = []
    intermediate_steps_list = instance.get("intermediate_steps", [])
    number_of_intermediate_steps = len(intermediate_steps_list)

    for idx in range(step):
      ho_input = intermediate_steps_list[idx][0]
      observation = intermediate_steps_list[idx][1]
      history.append({'action': ho_input[0], 'action_input': ho_input[1], 'observation': observation})

    system_text = SYSTEM_PROMPT.format(tool_docs=tool.get("NLDocumentation", "").strip())

    user_query = f"Query: {instance.get("input", "").strip()}\nHistory: {json.dumps(history)}"

    if step<number_of_intermediate_steps:
      trajectory = parse_output_trajectory_reasoning(intermediate_steps_list[step]).strip()

    else:
      trajectory = parse_output_trajectory_answer(instance).strip()


    return [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": trajectory},
    ]

In [ ]:
train_texts = []

for tool in raw_train:
    for instance in (tool.get("Instances", [])):
      for step in range(len(instance.get("intermediate_steps", []))+1):
        ex = make_training_example(tool, instance, step)
        if ex:
            train_texts.append({"messages": ex})

In [ ]:
print(train_texts[1]["messages"][1]["content"])

Query: I'm troubleshooting some requests, so can you help me send a POST request to https://httpbin.org/post with the header "Content-Type: application/json" and the data '{"name": "John Doe", "email": "john.doe@example.com"}'? Let me know the response details.
History: [{"action": "sendHttpRequest", "action_input": "{\"method\": \"POST\", \"url\": \"https://httpbin.org/post\", \"headers\": {\"Content-Type\": \"application/json\"}, \"data\": {\"name\": \"John Doe\", \"email\": \"john.doe@example.com\"}}", "observation": "Status Code: 200. Response: {\"response\":{\"status_code\":200,\"headers\":{\"Content-Type\":\"application/json\"},\"body\":\"{\\\"args\\\":{\\\"method\\\":\\\"POST\\\",\\\"url\\\":\\\"https://httpbin.org/post\\\",\\\"headers\\\":{\\\"'Content-Type'\\\":\\\"'application/json'\\\"},\\\"data\\\":{\\\"'name'\\\":\\\"'John Doe'\\\",\\\"'email'\\\":\\\"'john.doe@example.com'\\\"}},\\\"data\\\":\\\"\\\",\\\"files\\\":{},\\\"form\\\":{},\\\"headers\\\":{\\\"Accept\\\":\\\"*/*

In [ ]:
len(train_texts)

11970

In [ ]:
hf_dataset = Dataset.from_list(train_texts).shuffle(seed=42)
split = hf_dataset.train_test_split(test_size=0.2, seed=42)

train_ds, val_ds = split["train"], split["test"]

In [ ]:
len(train_ds), len(val_ds)

(9576, 2394)

In [ ]:
# cache_dir = "/content/drive/MyDrive/hf_cache"
# os.makedirs(cache_dir, exist_ok=True)

# os.environ["HF_HOME"] = cache_dir
# os.environ["TRANSFORMERS_CACHE"] = cache_dir

In [ ]:
model_kwargs = dict(
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear"
)

In [ ]:
args = SFTConfig(
    output_dir="gemma-4-ToolAlpaca-FineTune-C",
    max_length=MAX_SEQ_LEN,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=5e-5,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    push_to_hub=True,
    report_to="tensorboard",
    dataloader_num_workers=2,
    packing = False,
    gradient_accumulation_steps=4,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False
    },
    gradient_checkpointing=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/9576 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2394 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss
1,0.044232,0.047019


TrainOutput(global_step=1197, training_loss=0.3381721715978911, metrics={'train_runtime': 15368.8549, 'train_samples_per_second': 0.623, 'train_steps_per_second': 0.078, 'total_flos': 3.474665697588019e+16, 'train_loss': 0.3381721715978911})

In [ ]:
trainer.save_model()

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eTune-C/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...47530.e77c384019a8.1160.0: 100%|##########| 56.0kB / 56.0kB            

  ...adapter_model.safetensors: 100%|##########| 19.1MB / 19.1MB            

  ...FineTune-C/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
tokenizer.save_pretrained("gemma-4-ToolAlpaca-FineTune-C")

('gemma-4-ToolAlpaca-FineTune-C/tokenizer_config.json',
 'gemma-4-ToolAlpaca-FineTune-C/chat_template.jinja',
 'gemma-4-ToolAlpaca-FineTune-C/tokenizer.json')

In [ ]:
torch.cuda.empty_cache()

In [ ]:
trainer.args.per_device_eval_batch_size = 1
trainer.args.eval_accumulation_steps = 1
trainer.args.prediction_loss_only = True

metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch
0.044232,0.047019,1


{'eval_loss': 0.047019220888614655}


In [ ]:
from google.colab import runtime
runtime.unassign()